In [2]:
import numpy as np
import pandas as pd

In [4]:
y = pd.read_csv('../data/processed/transformed_data.csv', header=[0,1], index_col=0)
y

tenor             1                                                        \
moneyness      80.0     90.0     95.0     97.5    100.0    102.5    105.0   
Dates                                                                       
2016-04-01  31.4422  20.4818  15.7749  13.5518  11.1926   8.7991   8.6680   
2016-04-04  32.3816  21.0527  16.4633  14.4232  12.1854   9.7695   9.1445   
2016-04-05  32.8637  22.0677  17.6430  15.6324  13.4519  11.2138   9.7895   
2016-04-06  32.3115  20.9377  16.4275  14.4957  12.2162   9.9239   8.8714   
2016-04-07  33.5882  22.9630  18.5421  16.4382  14.1363  11.4923   9.7613   
...             ...      ...      ...      ...      ...      ...      ...   
2026-05-19  36.3007  24.9829  20.1980  17.9325  15.3420  13.4098  12.3552   
2026-05-20  35.9650  24.5599  19.5751  17.2373  15.0037  13.4033  12.5152   
2026-05-21  36.2621  24.3449  19.1241  16.7278  14.4524  12.9001  12.1053   
2026-05-22  36.9736  24.5802  19.2580  16.8579  14.5497  13.0436  12.3963   
2026-05-25  36.9736  24.5802  19.2580  16.8579  14.5497  13.0436  12.3963   

tenor                               2  ...       18       24           \
moneyness     110.0    120.0     80.0  ...    120.0     80.0     90.0   
Dates                                  ...                              
2016-04-01  12.3428  13.0078  26.5925  ...  11.8507  22.6374  20.1359   
2016-04-04  12.4163  13.2117  27.3005  ...  12.0997  22.8203  20.3300   
2016-04-05  12.3206  12.3206  28.1957  ...  12.5537  23.2521  20.7511   
2016-04-06  11.5959  11.5959  27.3145  ...  12.2230  22.9935  20.4688   
2016-04-07  11.6225  11.7970  28.9461  ...  12.8713  23.5757  21.0630   
...             ...      ...      ...  ...      ...      ...      ...   
2026-05-19  12.9116  19.7180  32.6861  ...  14.5225  24.6998  22.3562   
2026-05-20  13.4071  20.2456  32.2883  ...  14.5506  24.5622  22.2231   
2026-05-21  13.1761  20.5820  31.8372  ...  14.4419  24.4429  22.1017   
2026-05-22  13.9258  21.1173  31.8843  ...  14.5066  24.4502  22.1069   
2026-05-25  13.9258  21.1173  31.8843  ...  14.5066  24.4502  22.1069   

tenor                                                                      
moneyness      95.0     97.5    100.0    102.5    105.0    110.0    120.0  
Dates                                                                      
2016-04-01  18.9134  18.3122  17.6810  17.0812  16.4947  15.3507  13.3202  
2016-04-04  19.1046  18.4937  17.8915  17.2888  16.7016  15.5572  13.5400  
2016-04-05  19.5040  18.8611  18.3484  17.7543  17.1416  15.9772  13.9185  
2016-04-06  19.2367  18.6073  18.0355  17.4445  16.8486  15.6908  13.6485  
2016-04-07  19.8167  19.1972  18.6367  18.0572  17.4823  16.3173  14.2314  
...             ...      ...      ...      ...      ...      ...      ...  
2026-05-19  21.1966  20.6092  18.9822  18.4450  17.9264  16.9253  15.2512  
2026-05-20  21.0593  20.4780  19.3770  18.3458  17.8333  16.8508  15.2362  
2026-05-21  20.9393  20.3690  18.8140  18.2213  17.7088  16.7347  15.1386  
2026-05-22  20.9575  20.3814  19.3243  18.2352  17.7266  16.7621  15.1779  
2026-05-25  20.9575  20.3814  19.3243  18.2352  17.7266  16.7621  15.1779  

[2647 rows x 63 columns]

In [5]:
n = len(y)
y_train = y.iloc[:int(n*0.8)].to_numpy().astype(float)
y_test = y.iloc[int(n*0.8):].to_numpy().astype(float)

print(f"Total: {n}, train: {len(y_train)}, test: {len(y_test)}")

Total: 2647, train: 2117, test: 530


In [6]:
from statsmodels.tsa.api import VAR
from datetime import datetime
from collections import defaultdict

horizons = [1, 14, 30, 90, 180, 365]

class VAR_Results:
    def __init__(self, res):
        self.res = res
        self.p = res.k_ar
        self.forecasts = defaultdict(list)
        self.errors = defaultdict(list)
        self.mse = {}

    def compute_error(self, h: int, d: datetime, y: pd.DataFrame) -> None:
        """Compute forecasting error.
        
        Args:
            h: forecast horizon
            d: date to forecast
            y: dataset to backtest

        Return:
            Forecast error for the chosen date
        """
        d = y.index.get_loc(d)
        actual = y.iloc[d]
        inputs = y.iloc[d-h-self.p+1:d-h+1].to_numpy()
        pred = self.res.forecast(inputs, steps=h)[-1]
        self.forecasts[h].append(pred)
        self.errors[h].append((actual - pred)**2)
        
    def compute_mse(self, h) -> float:
        """Compute mean squared error from current sum of squared errors.

        Args:
            h: forecast horizon
        
        Return:
            Mean squared error
        """
        try:             
            self.mse[h] = np.mean(self.errors[h])
            return self.mse[h]
        
        except KeyError:
            print("There are no predictions for this forecast horizon.")

var = VAR(y_train)

In [7]:
print("Fitting VAR(1)...")
res_var_1 = VAR_Results(var.fit(1))

print("Fitting VAR(p) with AIC...")
res_var_aic = VAR_Results(var.fit(maxlags=20, ic='aic'))

print("Fitting VAR(p) with BIC...")
res_var_bic = VAR_Results(var.fit(maxlags=20, ic='bic'))

print(f"Lag order selected by AIC: {res_var_aic.p}")
print(f"Lag order selected by BIC: {res_var_bic.p}")

Fitting VAR(1)...
Fitting VAR(p) with AIC...
Fitting VAR(p) with BIC...
Lag order selected by AIC: 20
Lag order selected by BIC: 1


In [8]:
var_models = [res_var_1, res_var_aic, res_var_bic]

for h in horizons:
    for d in y.index[int(n*0.8):]:
        for model in var_models:
            model.compute_error(h, d, y)

    for model in var_models:
            model.compute_mse(h)

for model in var_models:
     print(f"{model.mse}")

{1: np.float64(0.8928109454679991), 14: np.float64(4.9550853866692846), 30: np.float64(5.984165571904562), 90: np.float64(6.269098987008175), 180: np.float64(5.18974318389112), 365: np.float64(5.6950103018931655)}
{1: np.float64(3.0887825421028214), 14: np.float64(16.633176359953918), 30: np.float64(16.978630324819413), 90: np.float64(16.49576405992234), 180: np.float64(15.860344517803828), 365: np.float64(15.221354375033398)}
{1: np.float64(0.8928109454679991), 14: np.float64(4.9550853866692846), 30: np.float64(5.984165571904562), 90: np.float64(6.269098987008175), 180: np.float64(5.18974318389112), 365: np.float64(5.6950103018931655)}
